<a href="https://colab.research.google.com/github/dbalsx/26-1-OSS-Project-team-19/blob/test_v1/%EC%9D%B8%EB%8F%84%EB%B3%B4%ED%96%89_%EC%98%81%EC%83%81_%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%85%8B_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 깃허브 저장소 복제
!git clone https://github.com/ChelseaGH/sidewalk_prototype_AI_Hub.git

# 2. 작업 폴더로 이동
%cd sidewalk_prototype_AI_Hub

# 3. YOLO 등 객체 인식에 필요한 기본 라이브러리 설치
!pip install ultralytics opencv-python

Cloning into 'sidewalk_prototype_AI_Hub'...
remote: Enumerating objects: 1201, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 1201 (delta 0), reused 0 (delta 0), pack-reused 1199 (from 1)
Receiving objects: 100% (1201/1201), 47.26 MiB | 16.48 MiB/s, done.
Resolving deltas: 100% (247/247), done.
/content/sidewalk_prototype_AI_Hub
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import cv2
import os

def video_to_frames(video_path, output_dir, frames_per_sec=2):
    """
    영상을 입력받아 원하는 초당 프레임 수(FPS)만큼 이미지를 추출하는 함수

    :param video_path: 구글 드라이브에 업로드한 영상 파일 경로
    :param output_dir: 추출된 이미지를 저장할 폴더 경로
    :param frames_per_sec: 1초당 추출할 이미지 장수 (기본값 2장)
    """

    # 1. 저장할 폴더가 없다면 생성
    os.makedirs(output_dir, exist_ok=True)

    # 2. OpenCV로 영상 파일 열기
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("에러: 영상 파일을 열 수 없습니다. 경로를 다시 확인해 주세요.")
        return

    # 영상의 원본 FPS (초당 프레임 수) 확인
    original_fps = round(cap.get(cv2.CAP_PROP_FPS))
    print(f"원본 영상 FPS: {original_fps}")

    # 몇 프레임마다 한 장씩 캡처할지 간격(Interval) 계산
    # 예: 원본이 30FPS이고 1초에 2장 뽑고 싶다면, 15프레임마다 1장씩 저장
    interval = max(1, original_fps // frames_per_sec)

    frame_count = 0
    saved_count = 0

    print("프레임 추출을 시작합니다...")

    # 3. 영상을 한 프레임씩 읽어가며 캡처
    while True:
        ret, frame = cap.read()

        # 영상이 끝나면(더 이상 읽을 프레임이 없으면) 루프 종료
        if not ret:
            break

        # 설정한 간격(interval)에 도달할 때만 이미지 저장
        if frame_count % interval == 0:
            # 파일명 지정 (예: frame_0000.jpg, frame_0015.jpg ...)
            filename = f"frame_{frame_count:04d}.jpg"
            save_path = os.path.join(output_dir, filename)

            # 이미지 파일로 저장
            cv2.imwrite(save_path, frame)
            saved_count += 1

        frame_count += 1

    # 4. 메모리 해제 및 종료
    cap.release()
    print(f"추출 완료! 총 {saved_count}장의 이미지가 {output_dir} 폴더에 저장되었습니다.")

# ==========================================
# 실제 실행 부분 (경로는 팀 프로젝트 폴더에 맞게 수정하세요)
# ==========================================

# 구글 드라이브에 업로드한 스마트폰 촬영 영상 경로 (예: .mp4, .mov)
video_file = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/sookmyung-2.mp4'

# 이미지가 저장될 새로운 폴더 경로
image_folder = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-2'

# 함수 실행 (1초당 2장씩 추출)
video_to_frames(video_file, image_folder, frames_per_sec=1)

원본 영상 FPS: 30
프레임 추출을 시작합니다...
추출 완료! 총 98장의 이미지가 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-2 폴더에 저장되었습니다.


In [5]:
# 1. 날아간 YOLOv8 패키지부터 다시 설치하기!
#!pip install ultralytics

from ultralytics import YOLO
import os

# 스마트폰 영상에서 추출한 내 이미지들이 모여있는 폴더 경로
image_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3'

# 이미지 학습이 완료된 똑똑한 기본 YOLO 나노 모델 불러오기
model = YOLO('yolov8n.pt')

# 예측(Predict)을 돌리면서 정답지 텍스트 파일과 시각화 이미지를 내 커스텀 폴더에 자동 저장
model.predict(
    source=image_dir,
    save_txt=True,
    save=True,  # AI가 친 박스를 눈으로 확인할 수 있게 이미지도 함께 저장!
    conf=0.25,
    project='/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset',
    name='hyochang-3_auto_labels'
)

print("🎉 AI가 초안 라벨링 및 시각화 이미지 저장을 모두 끝마쳤습니다!")


image 1/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0000.jpg: 384x640 (no detections), 7.3ms
image 2/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0030.jpg: 384x640 1 car, 8.7ms
image 3/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0060.jpg: 384x640 1 car, 6.8ms
image 4/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0090.jpg: 384x640 1 car, 15.0ms
image 5/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0120.jpg: 384x640 1 car, 7.0ms
image 6/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0150.jpg: 384x640 1 car, 6.5ms
image 7/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0180.jpg: 384x640 1 car, 6.4ms
image 8/287 /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3/frame_0210.jpg: 384x640 1 car, 1 giraffe, 6.6ms
image 9/287 /content/drive/MyDrive/26-1 오

In [ ]:
!unzip '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/2019-01-004.인도보행영상_sample.zip' -d '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test'

Archive:  /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/2019-01-004.인도보행영상_sample.zip
   creating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/bbox_sample.xml  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027451.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027452.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027453.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027454.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027455.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027456.jpg  
  inflating: /content/drive/MyDrive

In [ ]:
!mv /content/sidewalk_prototype_AI_Hub '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/'

In [ ]:
import os
import xml.etree.ElementTree as ET

# 1. 경로 설정 (1개짜리 통짜 XML 파일 경로로 직접 지정)
xml_file = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/bbox_sample.xml'
output_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/yolo_labels'

os.makedirs(output_dir, exist_ok=True)
classes = []

# 정규화 수학 공식 (그대로 사용)
def convert_coordinates(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[1]) / 2.0
    y_center = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    x_center = x_center * dw
    w = w * dw
    y_center = y_center * dh
    h = h * dh
    return (x_center, y_center, w, h)

print("단일 XML 파일 파싱을 시작합니다...")

# XML 파일 읽기
tree = ET.parse(xml_file)
root = tree.getroot()

# 2. <image> 태그들을 하나씩 돌면서 처리
image_count = 0
for image_tag in root.iter('image'):
    # 태그의 '속성(get)'에서 이름과 사이즈 가져오기
    image_name = image_tag.get('name')
    w_image = float(image_tag.get('width'))
    h_image = float(image_tag.get('height'))

    # 확장자(.jpg) 떼고 txt 파일명 만들기
    filename_without_ext = os.path.splitext(image_name)[0]
    txt_path = os.path.join(output_dir, filename_without_ext + '.txt')

    # 해당 이미지 안에 있는 장애물 <box> 찾기
    boxes = list(image_tag.iter('box'))

    # 장애물이 1개라도 있는 이미지일 경우에만 txt 파일 생성
    if len(boxes) > 0:
        image_count += 1
        with open(txt_path, 'w') as out_file:
            for box in boxes:
                cls_name = box.get('label')

                if cls_name not in classes:
                    classes.append(cls_name)
                cls_id = classes.index(cls_name)

                # AI 허브의 특이한 좌표 이름(xtl, xbr, ytl, ybr)에서 값 가져오기
                b = (float(box.get('xtl')), float(box.get('xbr')),
                     float(box.get('ytl')), float(box.get('ybr')))

                bb = convert_coordinates((w_image, h_image), b)
                out_file.write(str(cls_id) + " " + " ".join([str(a) for a in bb]) + '\n')

# 3. classes.txt 만들기
classes_file = os.path.join(output_dir, 'classes.txt')
with open(classes_file, 'w') as f:
    for cls in classes:
        f.write(cls + '\n')

print(f"변환 완료! 총 {image_count}개의 이미지에 대한 TXT 라벨이 생성되었습니다.")
print(f"총 {len(classes)}개의 장애물 클래스가 식별되었습니다.")

단일 XML 파일 파싱을 시작합니다...
변환 완료! 총 299개의 이미지에 대한 TXT 라벨이 생성되었습니다.
총 25개의 장애물 클래스가 식별되었습니다.


In [ ]:
import yaml
import os
import shutil
import glob

# 1. 경로 세팅
base_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test'
image_dir = os.path.join(base_dir, 'bbox')
label_dir = os.path.join(base_dir, 'yolo_labels')
yaml_path = os.path.join(base_dir, 'data.yaml')

# 2. YOLO가 라벨을 잘 찾게 하려면 이미지와 txt 파일이 같은 폴더에 있는 것이 제일 확실해!
# yolo_labels 폴더에 생성된 txt 파일들을 이미지(bbox) 폴더로 모두 복사
txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
for txt in txt_files:
    shutil.copy(txt, image_dir)
print("텍스트 라벨 파일들을 이미지 폴더로 성공적으로 합쳤습니다.")

# 3. classes.txt 읽어와서 클래스 목록 만들기
classes_file = os.path.join(label_dir, 'classes.txt')
with open(classes_file, 'r', encoding='utf-8') as f:
    # 빈 줄 제외하고 클래스 이름만 깔끔하게 리스트로 저장
    classes = [line.strip() for line in f.readlines() if line.strip()]

# 4. 우리가 원했던 data.yaml 내용 구성하기
data = {
    'train': image_dir,  # 학습용 이미지 경로
    'val': image_dir,    # 검증용 이미지 경로 (샘플이라 일단 동일하게!)
    'nc': len(classes),  # 클래스 총 개수
    'names': classes     # 클래스 이름 목록
}

# 5. data.yaml 파일로 예쁘게 저장
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, default_flow_style=None, allow_unicode=True)

print(f"data.yaml 생성 완료! 저장 위치: {yaml_path}")

텍스트 라벨 파일들을 이미지 폴더로 성공적으로 합쳤습니다.
data.yaml 생성 완료! 저장 위치: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml


In [ ]:
%cd '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝'

/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝


In [ ]:
!pip install ultralytics

In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data='/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml' epochs=10 imgsz=640 batch=16 project='/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/runs' name='sidewalk_sample_train'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, h

In [2]:
# 코랩 새 셀에 넣고 실행
!unzip -q '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3_bounding_V.zip' -d '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3_extracted'
print("🎉 압축 해제 완료!")

🎉 압축 해제 완료!


In [6]:
import os
import glob
import random
import shutil

# 경로 설정 (기존과 동일)
src_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/hyochang-3_extracted'
base_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/dataset_final'

train_img_dir = os.path.join(base_dir, 'images/train')
val_img_dir = os.path.join(base_dir, 'images/val')
train_lbl_dir = os.path.join(base_dir, 'labels/train')
val_lbl_dir = os.path.join(base_dir, 'labels/val')

# 폴더 생성
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

# [업그레이드] 하위 폴더까지 샅샅이 뒤져서 대소문자 상관없이 모든 이미지 찾기
images = []
for root, dirs, files in os.walk(src_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            images.append(os.path.join(root, file))

print(f"🔍 발견된 총 이미지 개수: {len(images)}장")

if len(images) == 0:
    print("❌ 경고: 지정한 폴더에서 이미지를 단 한 장도 찾지 못했습니다.")
    print(f"현재 경로인 '{src_dir}' 폴더 안에 진짜 사진 파일들이 들어있는지 구글 드라이브에서 직접 확인해 주세요!")
else:
    # 데이터 무작위 섞기 및 8:2 분할
    random.shuffle(images)
    split_idx = int(len(images) * 0.8)
    train_images = images[:split_idx]
    val_images = images[split_idx:]

    def move_files(image_list, dest_img, dest_lbl):
        count = 0
        for img_path in image_list:
            # 이미지 복사
            shutil.copy(img_path, dest_img)
            # 이미지와 같은 방에 있는 짝꿍 텍스트 파일 찾아서 복사
            txt_path = os.path.splitext(img_path)[0] + '.txt'
            if os.path.exists(txt_path):
                shutil.copy(txt_path, dest_lbl)
            count += 1
        return count

    train_count = move_files(train_images, train_img_dir, train_lbl_dir)
    val_count = move_files(val_images, val_img_dir, val_lbl_dir)

    print("\n📂 dataset_final 폴더에 8:2 비율로 데이터 분할 완료!")
    print(f"   - 학습용(Train) 이미지: {train_count}장")
    print(f"   - 검증용(Val) 이미지: {val_count}장")

🔍 발견된 총 이미지 개수: 287장

📂 dataset_final 폴더에 8:2 비율로 데이터 분할 완료!
   - 학습용(Train) 이미지: 229장
   - 검증용(Val) 이미지: 58장


In [7]:
import yaml

yaml_path = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/custom_data.yaml'

# ⚠️ 이미지에 나온 classes.txt의 순서와 대소문자를 100% 똑같이 맞춘 리스트야!
final_classes = [
    'dog', 'person', 'cat', 'tv', 'car',
    'meatballs', 'marinara sauce', 'tomato soup', 'chicken noodle soup', 'french onion soup',
    'chicken breast', 'ribs', 'pulled pork', 'hamburger', 'cavity',
    'bench', 'backpack', 'moto cycle', 'obs', 'stair'
]

data = {
    'train': '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/dataset_final/images/train',
    'val': '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/dataset_final/images/val',
    'nc': len(final_classes), # 알아서 20개로 잡혀!
    'names': final_classes
}

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, allow_unicode=True)

print("👍 이상한 클래스들까지 인덱스 안 깨지게 custom_data.yaml 생성 완료!")

👍 이상한 클래스들까지 인덱스 안 깨지게 custom_data.yaml 생성 완료!


In [8]:
import os
import shutil

# ==================== [1. 경로 설정] ====================
# ⚠️ 각 폴더의 진짜 구글 드라이브 경로가 맞는지 눈으로 한 번만 확인해 줘!
team_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/hyochang3.yolov5pytorch'
user_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/dataset_final'
master_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/custom_dataset/dataset_master'

# 마스터 서브 폴더 경로 정의
train_img_dst = os.path.join(master_dir, 'images/train')
val_img_dst = os.path.join(master_dir, 'images/val')
train_lbl_dst = os.path.join(master_dir, 'labels/train')
val_lbl_dst = os.path.join(master_dir, 'labels/val')

# 마스터 폴더 자동 생성
for d in [train_img_dst, val_img_dst, train_lbl_dst, val_lbl_dst]:
    os.makedirs(d, exist_ok=True)

# ==================== [2. 데이터 복사 함수 정의] ====================

def merge_teammate_data(src_split_dir, dst_img, dst_lbl, prefix):
    """ 로보플로우 구조(train/valid)에서 이미지와 txt를 찾아 마스터로 복사하는 함수 """
    if not os.path.exists(src_split_dir):
        return 0

    # 먼저 해당 분할 폴더 내의 모든 txt 파일 위치 맵핑
    txt_map = {}
    for root, _, files in os.walk(src_split_dir):
        for f in files:
            if f.lower().endswith('.txt') and f != 'classes.txt':
                txt_map[os.path.splitext(f)[0]] = os.path.join(root, f)

    # 이미지 파일을 찾아서 짝꿍 txt와 함께 복사
    copied_count = 0
    for root, _, files in os.walk(src_split_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, f)
                base, ext = os.path.splitext(f)

                # 팀원 간 파일명 중복 충돌 방지를 위해 이름 뒤에 고유 접미사(prefix) 추가
                dst_img_name = f"{base}_{prefix}{ext}"
                dst_txt_name = f"{base}_{prefix}.txt"

                # 최종 마스터 경로 복사
                shutil.copy(img_path, os.path.join(dst_img, dst_img_name))
                if base in txt_map:
                    shutil.copy(txt_map[base], os.path.join(dst_lbl, dst_txt_name))
                copied_count += 1
    return copied_count

def merge_user_data(src_img_dir, src_lbl_dir, dst_img, dst_lbl, prefix):
    """ 정석 구조(images/labels)에서 이미지와 txt를 찾아 마스터로 복사하는 함수 """
    if not os.path.exists(src_img_dir):
        return 0

    copied_count = 0
    for f in os.listdir(src_img_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            base, ext = os.path.splitext(f)
            img_path = os.path.join(src_img_dir, f)
            txt_path = os.path.join(src_lbl_dir, base + '.txt')

            dst_img_name = f"{base}_{prefix}{ext}"
            dst_txt_name = f"{base}_{prefix}.txt"

            shutil.copy(img_path, os.path.join(dst_img, dst_img_name))
            if os.path.exists(txt_path):
                shutil.copy(txt_path, os.path.join(dst_lbl, dst_txt_name))
            copied_count += 1
    return copied_count

# ==================== [3. 진짜 합체 작업 실행] ====================
print("🔄 뒤죽박죽 데이터셋 통합을 시작합니다...")

# 1. 팀원 폴더 통합 (로보플로우 구조 파싱)
# 팀원의 valid 데이터는 마스터의 val 폴더로 매핑해 줍니다.
team_train_cnt = merge_teammate_data(os.path.join(team_dir, 'train'), train_img_dst, train_lbl_dst, 'team')
team_val_cnt = merge_teammate_data(os.path.join(team_dir, 'valid'), val_img_dst, val_lbl_dst, 'team')

# 2. 내 폴더 통합 (정석 구조 파싱)
user_train_cnt = merge_user_data(os.path.join(user_dir, 'images/train'), os.path.join(user_dir, 'labels/train'), train_img_dst, train_lbl_dst, 'user')
user_val_cnt = merge_user_data(os.path.join(user_dir, 'images/val'), os.path.join(user_dir, 'labels/val'), val_img_dst, val_lbl_dst, 'user')

print("\n🎉 마스터 데이터셋 통합 완료!")
print(f"   - 마스터 학습용(Train) 이미지: 팀원 {team_train_cnt}장 + 내꺼 {user_train_cnt}장 합체 완료")
print(f"   - 마스터 검증용(Val) 이미지: 팀원 {team_val_cnt}장 + 내꺼 {user_val_cnt}장 합체 완료")

🔄 뒤죽박죽 데이터셋 통합을 시작합니다...

🎉 마스터 데이터셋 통합 완료!
   - 마스터 학습용(Train) 이미지: 팀원 201장 + 내꺼 228장 합체 완료
   - 마스터 검증용(Val) 이미지: 팀원 57장 + 내꺼 58장 합체 완료
